In [15]:
"""
fetch_tslt_options.py
=====================================================================
One-time fetch of TSLT option daily bars from Databento (OPRA.PILLAR),
transformed to match the existing TSLA_calls_close.parquet schema:
    columns: date, expiry, strike, px_last, px_volume   (calls only)

SAFETY: the paid data pull is gated behind CONFIRM_PULL.
Run the steps IN ORDER, read each checkpoint, and only set
CONFIRM_PULL = True after you've seen the cost estimate.

Confirmed from Databento docs:
    - dataset  = "OPRA.PILLAR"
    - schema   = "ohlcv-1d"   (daily OHLCV bars; cheap, EOD only)
    - history starts 2023-03-28
Reconstructed (VERIFY against Databento docs before relying on):
    - stype_in="parent" with symbol "TSLT.OPT" to grab the whole chain
    - client.metadata.get_cost(...) for the cost estimate
    - definition schema field names (raw_symbol, expiration,
      strike_price, instrument_class)
"""

'\nfetch_tslt_options.py\n=====================================================================\nOne-time fetch of TSLT option daily bars from Databento (OPRA.PILLAR),\ntransformed to match the existing TSLA_calls_close.parquet schema:\n    columns: date, expiry, strike, px_last, px_volume   (calls only)\n\nSAFETY: the paid data pull is gated behind CONFIRM_PULL.\nRun the steps IN ORDER, read each checkpoint, and only set\nCONFIRM_PULL = True after you\'ve seen the cost estimate.\n\nConfirmed from Databento docs:\n    - dataset  = "OPRA.PILLAR"\n    - schema   = "ohlcv-1d"   (daily OHLCV bars; cheap, EOD only)\n    - history starts 2023-03-28\nReconstructed (VERIFY against Databento docs before relying on):\n    - stype_in="parent" with symbol "TSLT.OPT" to grab the whole chain\n    - client.metadata.get_cost(...) for the cost estimate\n    - definition schema field names (raw_symbol, expiration,\n      strike_price, instrument_class)\n'

In [25]:
import databento as db
import pandas as pd

API_KEY = "db-VpTPeUDQ6KFtpemMPb7phChLt7p9c"      # <-- paste your Databento key
START   = "2023-10-19"             # TSLT inception (options may start later)
END     = "2026-06-11"             # exclusive-style bound; matches frozen window
OUT     = "TSLT_calls_close.parquet"

CONFIRM_PULL = False              # <-- leave False until you've seen the cost

client = db.Historical(key=API_KEY)

In [19]:
# ---------------------------------------------------------------------
# STEP 1 — DEFINITIONS (cheap): what TSLT contracts exist, and from when?
# This gives expiry / strike / call-or-put per instrument_id, which we
# need later to label the OHLCV bars.
# ---------------------------------------------------------------------
def step1_definitions():
    defs = client.timeseries.get_range(
        dataset="OPRA.PILLAR",
        schema="definition",
        symbols=["TSLT.OPT"],
        stype_in="parent",
        start=START,
        end=END,
    )
    d = defs.to_df()
    print("=== STEP 1: TSLT option definitions ===")
    print("Rows:", len(d))
    if len(d):
        cols = [c for c in ["raw_symbol", "expiration", "strike_price",
                            "instrument_class", "instrument_id"] if c in d.columns]
        print(d[cols].head(10).to_string())
        # earliest expiry / how far back contracts are defined
        if "expiration" in d.columns:
            print("\nExpiry range:", d["expiration"].min(), "->", d["expiration"].max())
        print("\ninstrument_class values:", d["instrument_class"].unique()
              if "instrument_class" in d.columns else "n/a")
    return d

In [21]:
# ---------------------------------------------------------------------
# STEP 2 — COST ESTIMATE (free): how many $ will the OHLCV pull cost?
# READ THIS before doing anything else. Expect a small fraction of $125
# for one underlying's chain at daily resolution.
# ---------------------------------------------------------------------
def step2_cost():
    cost = client.metadata.get_cost(
        dataset="OPRA.PILLAR",
        schema="ohlcv-1d",
        symbols=["TSLT.OPT"],
        stype_in="parent",
        start=START,
        end=END,
    )
    print("=== STEP 2: estimated cost ===")
    print(f"Estimated cost of the ohlcv-1d pull: ${cost}")
    print("If this is comfortably under your remaining credit, set "
          "CONFIRM_PULL = True and run step3.")
    return cost

In [23]:
# ---------------------------------------------------------------------
# STEP 3 — THE PAID PULL (gated). Pull daily bars, join to definitions
# to attach expiry/strike/type, filter to calls, save in TSLA schema.
# ---------------------------------------------------------------------
def step3_pull_and_save(defs_df):
    if not CONFIRM_PULL:
        print("CONFIRM_PULL is False — skipping paid pull. "
              "Set it to True once you've checked the cost.")
        return None

    bars = client.timeseries.get_range(
        dataset="OPRA.PILLAR",
        schema="ohlcv-1d",
        symbols=["TSLT.OPT"],
        stype_in="parent",
        start=START,
        end=END,
    )
    b = bars.to_df()
    print("=== STEP 3: raw ohlcv-1d bars ===")
    print("Rows:", len(b))
    print("Columns:", list(b.columns))

    # --- attach expiry / strike / class via instrument_id join ---
    # definitions carry the contract metadata; bars carry price+volume.
    key = "instrument_id"
    meta_cols = [c for c in ["instrument_id", "expiration",
                             "strike_price", "instrument_class"] if c in defs_df.columns]
    meta = defs_df[meta_cols].drop_duplicates(subset=[key])
    merged = b.reset_index().merge(meta, on=key, how="left")

    # --- filter to CALLS only ---
    # instrument_class is typically 'C' for calls, 'P' for puts.
    if "instrument_class" in merged.columns:
        merged = merged[merged["instrument_class"].astype(str).str.upper().str.startswith("C")]

    # --- build the target schema: date, expiry, strike, px_last, px_volume ---
    # ts_event (the bar timestamp) -> date ; close -> px_last ; volume -> px_volume
    date_col  = "ts_event" if "ts_event" in merged.columns else merged.columns[0]
    out = pd.DataFrame({
        "date":      pd.to_datetime(merged[date_col]).dt.tz_localize(None).dt.normalize(),
        "expiry":    pd.to_datetime(merged["expiration"]).dt.tz_localize(None).dt.normalize(),
        "strike":    merged["strike_price"].astype(float),
        "px_last":   merged["close"].astype(float),
        "px_volume": merged["volume"].astype(float),
    })

    # --- SANITY CHECKS before saving ---
    print("\n=== sanity checks ===")
    print("Strike range:", out["strike"].min(), "->", out["strike"].max(),
          "(TSLT ~$20, so strikes should be tens of dollars, NOT thousands)")
    print("Date range:", out["date"].min().date(), "->", out["date"].max().date())
    print("Unique expiries:", out["expiry"].nunique())
    print("Unique trading days:", out["date"].nunique())
    print("Rows after calls-only filter:", len(out))

    out = out.sort_values(["date", "expiry", "strike"]).reset_index(drop=True)
    out.to_parquet(OUT)
    print(f"\nSaved -> {OUT}")
    return out


if __name__ == "__main__":
    defs_df = step1_definitions()
    print("\n" + "-"*60 + "\n")
    step2_cost()
    print("\n" + "-"*60 + "\n")
    step3_pull_and_save(defs_df)

C:\Users\balod\AppData\Local\Temp\ipykernel_37824\2296106750.py:7: BentoWarning: The streaming request contained one or more days which have reduced quality: 2024-06-03 (degraded), 2025-10-22 (degraded). See: https://databento.com/docs/api-reference-historical/metadata/metadata-get-dataset-condition
  defs = client.timeseries.get_range(


=== STEP 1: TSLT option definitions ===
Rows: 219354
                                                raw_symbol                expiration  strike_price instrument_class  instrument_id
ts_recv                                                                                                                           
2023-11-06 14:25:00.017847813+00:00  TSLT  240315C00024000 2024-03-15 00:00:00+00:00          24.0                C      755032120
2023-11-06 14:25:00.017847813+00:00  TSLT  231215P00018000 2023-12-15 00:00:00+00:00          18.0                P      755032122
2023-11-06 14:25:00.018429393+00:00  TSLT  240315C00013000 2024-03-15 00:00:00+00:00          13.0                C      755032124
2023-11-06 14:25:00.018429393+00:00  TSLT  240621P00028000 2024-06-21 00:00:00+00:00          28.0                P      755032123
2023-11-06 14:25:00.054896641+00:00  TSLT  240315P00016000 2024-03-15 00:00:00+00:00          16.0                P      755032170
2023-11-06 14:25:00.056039215+

C:\Users\balod\AppData\Local\Temp\ipykernel_37824\241328683.py:11: BentoWarning: The streaming request contained one or more days which have reduced quality: 2024-06-03 (degraded), 2025-10-22 (degraded). See: https://databento.com/docs/api-reference-historical/metadata/metadata-get-dataset-condition
  bars = client.timeseries.get_range(


=== STEP 3: raw ohlcv-1d bars ===
Rows: 65027
Columns: ['rtype', 'publisher_id', 'instrument_id', 'open', 'high', 'low', 'close', 'volume', 'symbol']

=== sanity checks ===
Strike range: 1.0 -> 80.0 (TSLT ~$20, so strikes should be tens of dollars, NOT thousands)
Date range: 2023-11-06 -> 2026-06-10
Unique expiries: 37
Unique trading days: 649
Rows after calls-only filter: 38909

Saved -> TSLT_calls_close.parquet


In [27]:
tslt_opt = pd.read_parquet("TSLT_calls_close.parquet")

# How many distinct call contracts trade per day, on average?
per_day = tslt_opt.groupby("date").agg(
    n_contracts=("strike", "size"),
    n_strikes=("strike", "nunique"),
    n_expiries=("expiry", "nunique"),
)
print(per_day.describe())

# Spot-check a recent day
d = tslt_opt["date"].max()
print(f"\nContracts on {d.date()}:", len(tslt_opt[tslt_opt.date==d]))

       n_contracts   n_strikes  n_expiries
count   649.000000  649.000000  649.000000
mean     59.952234   17.727273    7.016949
std      40.024829    7.913764    3.121711
min       1.000000    1.000000    1.000000
25%      30.000000   12.000000    4.000000
50%      50.000000   17.000000    7.000000
75%      80.000000   23.000000    9.000000
max     241.000000   47.000000   19.000000

Contracts on 2026-06-10: 28
